In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import plotly.express as px

matplotlib.rcParams['font.family'] = 'AppleGothic', 'Malgun Gothic', 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

illegal_summary = pd.read_csv('../data/processed/illegal_summary.csv')
parking_summary = pd.read_csv('../data/processed/parking_summary.csv')

print(illegal_summary.shape)
print(parking_summary.shape)

display(illegal_summary.head())
display(parking_summary.head())

(29, 4)
(29, 3)


,지역,단속건수,면적,면적 당 단속건수
0,새롬동,7922,1.49,5317
1,다정동,8908,1.70,5240
2,아름동,10572,2.19,4827
3,어진동,11016,2.65,4157
4,고운동,15164,5.35,2834


,지역,주차장수,주차구획수
0,조치원읍,38,1299
1,나성동,4,882
2,보람동,4,238
3,아름동,1,236
4,종촌동,1,160


In [2]:

# 지역을 기준으로 데이터 병합
priority_df = pd.merge(
    illegal_summary,
    parking_summary,
    on='지역',
    how='inner'
)

priority_df

,지역,단속건수,면적,면적 당 단속건수,주차장수,주차구획수
0,새롬동,7922,1.49,5317,0,0
1,다정동,8908,1.70,5240,0,0
2,아름동,10572,2.19,4827,1,236
3,어진동,11016,2.65,4157,0,0
4,고운동,15164,5.35,2834,0,0
5,대평동,4065,1.50,2710,0,0
6,소담동,3086,1.17,2638,1,51
7,보람동,3352,1.33,2520,4,238
8,도담동,4480,2.03,2207,1,13
9,조치원읍,28321,13.56,2089,38,1299


주차구획 1면당 단속건수
= 주차공간 1칸당 불법주정차 단속이 얼마나 발생했는지

면적당 주차구획수
= 지역 면적 대비 주차공간이 얼마나 있는지

In [3]:
# 비교 지표 생성
import numpy as np

priority_df['주차구획 1면당 단속건수'] = np.where(
    priority_df['주차구획수'] > 0,
    (priority_df['단속건수'] / priority_df['주차구획수']).round(2),
    np.nan
)

priority_df['면적당 주차구획수'] = np.where(
    priority_df['면적'] > 0,
    (priority_df['주차구획수'] / priority_df['면적']).round(2),
    np.nan
)

In [4]:
priority_df = priority_df.sort_values(
    '주차구획 1면당 단속건수',
    ascending=False
)

priority_df

,지역,단속건수,면적,면적 당 단속건수,주차장수,주차구획수,주차구획 1면당 단속건수,면적당 주차구획수
8,도담동,4480,2.03,2207,1,13,344.62,6.40
19,연동면,944,21.50,44,1,10,94.40,0.47
11,한솔동,2308,2.70,855,1,33,69.94,12.22
6,소담동,3086,1.17,2638,1,51,60.51,43.59
2,아름동,10572,2.19,4827,1,236,44.80,107.76
9,조치원읍,28321,13.56,2089,38,1299,21.80,95.80
7,보람동,3352,1.33,2520,4,238,14.08,178.95
22,금남면,1219,72.55,17,2,96,12.70,1.32
13,나성동,9780,24.91,393,4,882,11.09,35.41
10,종촌동,1547,1.15,1345,1,160,9.67,139.13


In [5]:
fig = px.scatter(
    priority_df,
    x='주차구획수',
    y='단속건수',
    size='면적 당 단속건수',
    text='지역',
    title='주차구획수와 불법주정차 단속건수 비교',
    labels={
        '주차구획수': '주차구획수',
        '단속건수': '불법주정차 단속건수',
        '면적 당 단속건수': '면적 당 단속건수'
    },
    hover_data=[
        '지역',
        '주차장수',
        '주차구획 1면당 단속건수',
        '면적당 주차구획수'
    ]
)

fig.update_traces(textposition='top center')
fig.show()

| 지역   |                                                       |
| ---- | --------------------------------------------------------- |
| 조치원읍 | 주차구획수와 단속건수가 모두 가장 높아, 지역 규모나 유동인구의 영향이 큰 지역으로 볼 수 있음     |
| 나성동  | 주차구획수가 많음에도 단속건수가 높아, 주차장 접근성이나 상업지역 특성 등을 추가로 고려할 필요가 있음 |
| 아름동  | 주차구획수 대비 단속건수가 높고 점 크기도 커, 면적 대비 단속이 집중된 지역으로 볼 수 있음      |
| 도담동  | 주차구획수는 적은 편이나 단속건수가 비교적 높아, 주차공급 부족 가능성을 검토할 수 있음         |
| 보람동  | 주차구획수와 단속건수가 중간 수준으로 나타나, 일정 수준의 주차 문제가 존재하는 지역으로 볼 수 있음  |
| 한솔동  | 주차구획수와 단속건수가 모두 낮은 편이나, 주차공간 대비 단속 정도는 추가 확인이 필요함         |
| 금남면  | 주차구획수와 단속건수가 모두 낮아, 그래프상 주요 단속 집중 지역은 아님                  |
